# Corn Strategy Backtest Research

Standalone research notebook for corn. The strategy logic is inside this notebook: generic A/B tests, price Momentum/MR, annual walk-forward, Cargill overlay, and named-period check.

This is scoped like a one-week research pass: keep the feature set simple, compare a few understandable variants, and use Cargill data as a small physical filter around the price signal.


In [1]:
import pandas as pd
from IPython.display import Markdown, display

from corn_backtest import (
    abundant_supply_masks,
    candidate_comparison_table,
    candidate_metric_table,
    candidate_reference_row,
    evaluate_guarded_candidate_signals,
    evaluate_regime_family_signal_sets,
    evaluate_signal_candidates,
    momentum_reversal_signals,
    oos_metric_row,
    physical_disagreement_candidate_signals,
    walk_forward_signal_selection,
)
from corn_signals import (
    build_corn_product_flow_signal_universe,
    corn_dd_pct_table,
    corn_signal_set_families,
    load_train_set,
)
from shared_backtest import (
    backtest_positions_with_costs,
    build_feature_panels,
    clean_signal,
    period_metrics,
    positions_from_signal,
    signal_average,
)
from research_config import (
    CORN_HOLDING_COST_RATE,
    CORN_IC_THRESHOLD,
    CORN_MAX_ABS_LOT,
    CORN_TARGET_DAILY_PNL_VOL,
    CORN_TRADE_COST_PER_LOT,
    CORN_TRAIN_END,
    DEFAULT_MARGIN_PER_LOT,
    SPLIT_DATE,
)

pd.set_option("display.width", 180)
pd.set_option("display.max_columns", 60)
pd.set_option("display.float_format", lambda value: f"{value:,.3f}")

DATA_DIR = "train_set"
COMMODITY = "CORN"
TRAIN_END = pd.Timestamp(CORN_TRAIN_END)
OOS_START = pd.Timestamp(SPLIT_DATE)
DD_CAPITAL_USD = 10_000.0


## 1. Load Data And Confirm Cargill Inputs

Load the local train-set files, build corn feature panels, and confirm the Cargill crush/inventory inputs are present before testing signals.


In [2]:
data = load_train_set(DATA_DIR)
feature_panels, futures_pnl_all = build_feature_panels(data)
trading_index = futures_pnl_all.index
feature_panels = {name: panel.reindex(trading_index).fillna(0.0) for name, panel in feature_panels.items()}
futures_pnl = futures_pnl_all[[COMMODITY]].copy()
panel = feature_panels[COMMODITY]
signals = build_corn_product_flow_signal_universe(feature_panels, futures_pnl_all, DATA_DIR)
families_by_set = corn_signal_set_families(signals)

summary = pd.DataFrame([{
    "commodity": COMMODITY,
    "start": trading_index.min().date(),
    "end": trading_index.max().date(),
    "rows": len(trading_index),
    "corn_features": panel.shape[1],
    "signals": len(signals),
    "has_cargill_crush_activity": {"crush_surprise", "crush_utilization"}.issubset(panel.columns),
    "train_rows": int((trading_index < TRAIN_END).sum()),
    "validation_rows": int(((trading_index >= TRAIN_END) & (trading_index < OOS_START)).sum()),
    "oos_rows": int((trading_index >= OOS_START).sum()),
}])
coverage = pd.DataFrame([
    {
        "signal_set": signal_set,
        "family": family,
        "signals": len(members),
        "nonzero_signals": int(sum(series.abs().sum() > 0.0 for series in members.values())),
    }
    for signal_set, family_map in families_by_set.items()
    for family, members in family_map.items()
])

display(summary)
display(coverage)


,commodity,start,end,rows,corn_features,signals,has_cargill_crush_activity,train_rows,validation_rows,oos_rows
0,CORN,2010-01-04,2020-12-31,2866,21,18,True,1565,520,781


,signal_set,family,signals,nonzero_signals
0,A,prices,7,7
1,A,fundamentals,7,7
2,A,macro,2,2
3,B,prices,7,7
4,B,fundamentals,5,5
5,alpha,eia,1,1
6,alpha,macro,2,2
7,alpha,weather,1,1


## 2. Standalone Helpers

Small notebook helpers keep sizing, drawdown display, costs, and split metrics consistent across every candidate.


In [3]:
def with_dd_pct(table, columns=None):
    return corn_dd_pct_table(table, columns=columns, dd_capital_usd=DD_CAPITAL_USD)


def corn_positions(signal):
    return positions_from_signal(
        signal,
        futures_pnl,
        COMMODITY,
        target_daily_vol=CORN_TARGET_DAILY_PNL_VOL,
        max_abs_lot=CORN_MAX_ABS_LOT,
        halflife=2.0,
        threshold=0.05,
    )


def backtest_corn_positions(positions):
    return backtest_positions_with_costs(
        positions,
        futures_pnl,
        trade_cost_per_lot=CORN_TRADE_COST_PER_LOT,
        holding_cost_rate=CORN_HOLDING_COST_RATE,
        margin_per_lot=DEFAULT_MARGIN_PER_LOT,
    )[0]


def backtest_for_signal(signal):
    return backtest_corn_positions(corn_positions(signal))


def corn_abundant_supply_masks():
    return abundant_supply_masks(
        data["adj1"][COMMODITY].reindex(trading_index),
        panel["mom_60"],
        trading_index,
    )


## 3. Generic Signal A/B Tests

Start with the requested Signal A/B diagnostics. These are quick checks, not the final strategy: fixed recipes, select-by-IC, and a small online linear benchmark show whether fitting adds enough value to keep.


In [4]:
target = futures_pnl[COMMODITY].shift(-1)
generic = evaluate_regime_family_signal_sets(
    families_by_set,
    target,
    panel["mom_60"].abs(),
    trading_index,
    backtest_for_signal,
    TRAIN_END,
    OOS_START,
    CORN_IC_THRESHOLD,
)

display(with_dd_pct(generic["results"], [
    "signal_set", "strategy", "mode", "train_sharpe", "validation_sharpe",
    "oos_sharpe", "oos_pnl", "oos_dd", "full_sharpe", "turnover",
]))


,signal_set,strategy,mode,train_sharpe,validation_sharpe,oos_sharpe,oos_pnl,oos_dd_pct,full_sharpe,turnover
2,A,best_family_by_trend_mr,long_short,0.050,1.002,-0.791,-726.143,-10.228,-0.048,0.009
0,A,avg_all_signals,long_short,-0.279,0.627,0.206,146.475,-4.358,0.059,0.004
3,A,select_by_ic,long_short,0.263,0.361,-1.001,-562.334,-7.670,-0.107,0.005
1,A,equal_family,long_short,-0.164,0.340,0.665,421.742,-2.311,0.174,0.004
5,A,kalman_family_model,long_short,-0.299,0.169,-0.351,-591.673,-13.385,-0.212,0.012
4,A,expanding_ols_family_model,long_short,-0.279,-0.828,-0.481,-750.347,-14.613,-0.444,0.010
8,B,best_family_by_trend_mr,long_short,-0.065,1.034,-0.761,-732.246,-12.177,0.001,0.008
7,B,equal_family,long_short,-0.250,0.787,-0.154,-135.795,-7.708,0.011,0.005
6,B,avg_all_signals,long_short,-0.287,0.763,-0.032,-28.380,-7.218,0.018,0.005
9,B,select_by_ic,long_short,0.263,0.278,-0.418,-304.380,-7.340,0.083,0.007


## 4. Momentum/MR Price Benchmark

Use a simple price-only Momentum/MR benchmark as the spine. It tests pure momentum, short-term mean reversion, a fixed blend, and a trend/chop switch, then applies the same abundant-supply guards used later.


In [5]:
momentum_signals = momentum_reversal_signals(panel, trading_index)
momentum_eval = evaluate_signal_candidates(
    momentum_signals,
    backtest_for_signal,
    TRAIN_END,
    OOS_START,
    test="momentum_mr_benchmark",
    signal_set="price_only",
    source_table="momentum_mr_benchmark",
)
momentum_mr = {
    "signals": momentum_signals,
    "results": momentum_eval["results"],
    "supply_masks": corn_abundant_supply_masks(),
    "guard_results": momentum_eval["guard_results"],
}

display(Markdown("### Raw Momentum/MR benchmarks"))
display(with_dd_pct(momentum_mr["results"], [
    "strategy", "mode", "train_sharpe", "validation_sharpe", "oos_sharpe",
    "oos_pnl", "oos_dd", "full_sharpe", "full_dd", "turnover",
]))

display(Markdown("### Momentum/MR with same abundant-supply guards"))
display(with_dd_pct(momentum_mr["guard_results"], [
    "base_strategy", "guard", "oos_sharpe", "oos_pnl", "oos_dd",
    "full_sharpe", "full_dd", "turnover", "guard_oos_pct",
]).head(12))


### Raw Momentum/MR benchmarks

,strategy,mode,train_sharpe,validation_sharpe,oos_sharpe,oos_pnl,oos_dd_pct,full_sharpe,full_dd_pct,turnover
0,mom_20,long_short,0.264,-0.718,0.919,"1,552.034",-8.116,0.325,-14.411,0.011
1,mom_60,long_short,0.397,-0.854,0.518,989.913,-9.501,0.215,-19.786,0.008
4,trend_mom_else_mr,long_short,-0.029,-0.140,0.326,664.433,-12.827,0.068,-20.011,0.015
3,mom_60_rev_5_equal,long_short,0.124,0.036,0.198,223.511,-6.248,0.132,-12.788,0.012
2,rev_5,long_short,-0.404,1.106,-0.320,-421.342,-7.332,-0.125,-21.130,0.022


### Momentum/MR with same abundant-supply guards

,base_strategy,guard,oos_sharpe,oos_pnl,oos_dd_pct,full_sharpe,full_dd_pct,turnover,guard_oos_pct
0,mom_20,no_guard,0.919,"1,552.034",-8.116,0.325,-14.411,0.011,0.000
1,mom_60,no_guard,0.518,989.913,-9.501,0.215,-19.786,0.008,0.000
4,trend_mom_else_mr,no_guard,0.326,664.433,-12.827,0.068,-20.011,0.015,0.000
3,mom_60_rev_5_equal,no_guard,0.198,223.511,-6.248,0.132,-12.788,0.012,0.000
2,rev_5,no_guard,-0.320,-421.342,-7.332,-0.125,-21.130,0.022,0.000


## 5. Annual Walk-Forward Momentum/MR

Repeat the price benchmark with annual walk-forward selection. Each January selection uses only prior data, with the most recent two years treated as validation.


In [6]:
backtest_for_signal = lambda signal: backtest_corn_positions(corn_positions(signal))
walk_forward_signal, selected = walk_forward_signal_selection(momentum_mr["signals"], trading_index, backtest_for_signal, OOS_START)
walk_forward_bt = backtest_for_signal(walk_forward_signal)
static_bt = backtest_for_signal(momentum_mr["signals"]["mom_20"])
walk_forward = {
    "signal": walk_forward_signal,
    "selected": selected,
    "positions": corn_positions(walk_forward_signal),
    "backtest": walk_forward_bt,
    "comparison": pd.DataFrame([
        oos_metric_row("static_best_raw_momentum_mr_mom_20", static_bt, OOS_START),
        oos_metric_row("annual_walk_forward_momentum_mr", walk_forward_bt, OOS_START),
    ]),
}

display(Markdown("### Walk-forward Momentum/MR selected rows"))
display(with_dd_pct(walk_forward["selected"]))

display(Markdown("### Walk-forward Momentum/MR check"))
display(with_dd_pct(walk_forward["comparison"]))


### Walk-forward Momentum/MR selected rows

,rebalance,candidate,eligible,score,train_sharpe,validation_sharpe,validation_dd_pct,trade_sharpe,trade_pnl,trade_dd_pct,selected,selection_read
3,2018-01-01,mom_60_rev_5_equal,True,-0.545,0.124,0.036,-6.122,-0.675,-256.988,-4.716,True,Selected using only data before this rebalance...
2,2019-01-01,rev_5,False,-inf,-0.298,0.263,-7.332,0.393,169.021,-5.842,True,Fallback: no candidate passed positive train/v...
0,2020-01-01,mom_20,True,-0.091,0.055,0.707,-8.116,1.416,720.338,-3.680,True,Selected using only data before this rebalance...


### Walk-forward Momentum/MR check

,strategy,oos_sharpe,oos_pnl,oos_dd_pct,oos_active_days,full_sharpe,full_pnl,full_dd_pct
0,static_best_raw_momentum_mr_mom_20,0.919,"1,552.034",-8.116,698.000,0.325,"1,756.591",-14.411
1,annual_walk_forward_momentum_mr,0.488,643.137,-5.842,669.000,0.488,643.137,-5.842


## 6. Cargill Overlay And Final Candidate

Add Cargill as a small physical overlay/filter around the walk-forward Momentum/MR signal, then run the same guard menu. The goal is to improve risk control without making Cargill the whole alpha.


In [7]:
cargill_physical_signal = clean_signal(
    signal_average([signals["given_cgl_inventory_pressure"], signals["given_cgl_crush_activity"]], trading_index),
    trading_index,
)
base_spine_signals = {
    "wf_momentum_mr": walk_forward["signal"],
    "static_mom_20": momentum_mr["signals"]["mom_20"],
}
candidate_signals = physical_disagreement_candidate_signals(
    base_spine_signals,
    cargill_physical_signal,
    trading_index,
)
raw_results = candidate_metric_table(
    candidate_signals,
    backtest_for_signal,
    "cargill_overlay",
    OOS_START,
)
supply_masks = corn_abundant_supply_masks()
guard_results, guard_backtests = evaluate_guarded_candidate_signals(
    candidate_signals,
    supply_masks,
    corn_positions,
    backtest_corn_positions,
    COMMODITY,
    OOS_START,
    "cargill_overlay_guarded",
)
final_row = candidate_reference_row(
    guard_results,
    "wf_momentum_mr",
    sort_columns=("oos_sharpe", "full_sharpe", "oos_pnl"),
)
final_key = (final_row["base_strategy"], final_row["variant"], final_row["guard"])
cargill = {
    "cargill_physical_signal": cargill_physical_signal,
    "candidate_signals": candidate_signals,
    "raw_results": raw_results,
    "supply_masks": supply_masks,
    "guard_results": guard_results,
    "guard_backtests": guard_backtests,
    "final_row": final_row,
    "final_key": final_key,
}

display(Markdown("### Cargill overlay/filter raw candidates"))
display(with_dd_pct(cargill["raw_results"], [
    "base_strategy", "variant", "oos_sharpe", "oos_pnl", "oos_dd",
    "full_sharpe", "full_dd", "turnover",
]))

display(Markdown("### Cargill overlay/filter with abundant-supply guards"))
display(with_dd_pct(cargill["guard_results"], [
    "base_strategy", "variant", "guard", "oos_sharpe", "oos_pnl", "oos_dd",
    "full_sharpe", "full_dd", "turnover", "guard_oos_pct",
]).head(18))

display(Markdown("### Selected walk-forward Momentum/MR + Cargill candidate"))
display(with_dd_pct(pd.DataFrame([cargill["final_row"]]), [
    "base_strategy", "variant", "guard", "oos_sharpe", "oos_pnl", "oos_dd",
    "full_sharpe", "full_dd", "turnover", "guard_oos_pct",
]))


### Cargill overlay/filter raw candidates

,base_strategy,variant,oos_sharpe,oos_pnl,oos_dd_pct,full_sharpe,full_dd_pct,turnover
9,static_mom_20,cargill_disagree_flat,1.021,"1,151.909",-5.171,0.550,-10.412,0.013
8,static_mom_20,cargill_disagree_half,0.943,"1,360.747",-6.520,0.411,-11.452,0.011
6,static_mom_20,cargill_overlay_90_10,0.923,"1,423.178",-7.287,0.343,-12.602,0.011
5,static_mom_20,base_no_cargill,0.919,"1,552.034",-8.116,0.325,-14.411,0.011
7,static_mom_20,cargill_overlay_85_15,0.908,"1,337.512",-7.007,0.348,-12.119,0.010
4,wf_momentum_mr,cargill_disagree_flat,0.608,463.370,-3.420,0.608,-3.420,0.023
3,wf_momentum_mr,cargill_disagree_half,0.511,540.680,-4.336,0.511,-4.336,0.019
0,wf_momentum_mr,base_no_cargill,0.488,643.137,-5.842,0.488,-5.842,0.021
2,wf_momentum_mr,cargill_overlay_85_15,0.466,528.717,-5.227,0.381,-5.227,0.013
1,wf_momentum_mr,cargill_overlay_90_10,0.454,548.655,-5.452,0.392,-5.452,0.015


### Cargill overlay/filter with abundant-supply guards

,base_strategy,variant,guard,oos_sharpe,oos_pnl,oos_dd_pct,full_sharpe,full_dd_pct,turnover,guard_oos_pct
29,static_mom_20,cargill_disagree_flat,below_ma_or_negative_mom_flat,1.565,507.441,-2.453,0.881,-7.133,0.017,0.780
14,wf_momentum_mr,cargill_disagree_flat,below_ma_or_negative_mom_flat,1.413,215.627,-1.386,1.413,-1.386,0.028,0.780
17,static_mom_20,base_no_cargill,below_ma_or_negative_mom_flat,1.247,644.146,-3.698,0.542,-13.311,0.016,0.780
26,static_mom_20,cargill_disagree_half,below_ma_or_negative_mom_flat,1.194,539.440,-2.825,0.619,-10.305,0.015,0.780
20,static_mom_20,cargill_overlay_90_10,below_ma_or_negative_mom_flat,1.153,556.609,-3.480,0.554,-11.716,0.016,0.780
23,static_mom_20,cargill_overlay_85_15,below_ma_or_negative_mom_flat,1.095,507.122,-3.308,0.556,-10.819,0.015,0.780
27,static_mom_20,cargill_disagree_flat,no_guard,1.021,"1,151.909",-5.171,0.550,-10.412,0.013,0.000
28,static_mom_20,cargill_disagree_flat,below_ma_or_negative_mom_half,0.964,824.056,-2.586,0.538,-8.558,0.009,0.780
24,static_mom_20,cargill_disagree_half,no_guard,0.943,"1,360.747",-6.520,0.411,-11.452,0.011,0.000
18,static_mom_20,cargill_overlay_90_10,no_guard,0.923,"1,423.178",-7.287,0.343,-12.602,0.011,0.000


### Selected walk-forward Momentum/MR + Cargill candidate

,base_strategy,variant,guard,oos_sharpe,oos_pnl,oos_dd_pct,full_sharpe,full_dd_pct,turnover,guard_oos_pct
14,wf_momentum_mr,cargill_disagree_flat,below_ma_or_negative_mom_flat,1.413,215.627,-1.386,1.413,-1.386,0.028,0.780


## 7. Named-period Check

Check the final selected row against named market periods and summarize the recommendation.


In [8]:
final_row = cargill["final_row"]
final_key = cargill["final_key"]
final_periods = period_metrics(cargill["guard_backtests"][final_key])[
    ["period", "total_pnl", "sharpe", "max_drawdown", "hit_rate", "active_days", "note"]
]
wf_base_reference = candidate_reference_row(
    cargill["guard_results"],
    "wf_momentum_mr",
    variant="base_no_cargill",
    guard="no_guard",
)
mom20_guard_reference = candidate_reference_row(
    cargill["guard_results"],
    "static_mom_20",
    variant="base_no_cargill",
)
final_comparison = candidate_comparison_table([
    ("final_wf_momentum_mr_cargill", final_row),
    ("wf_momentum_mr_base_reference", wf_base_reference),
    ("guarded_mom20_benchmark", mom20_guard_reference),
])
conclusion = f"""
### Conclusion

**Core idea:** annual walk-forward Momentum/MR is the main tradable structure.

**Cargill use:** Cargill inventory/crush activity is used as a physical disagreement filter, not as the dominant corn alpha.

**Final corn candidate:** {final_row["base_strategy"]} with {final_row["variant"]} and guard {final_row["guard"]}. OOS Sharpe {final_row["oos_sharpe"]:.3f}, OOS PnL {final_row["oos_pnl"]:.3f}, OOS DD {100.0 * final_row["oos_dd"] / 10000.0:.2f}%, full-period Sharpe {final_row["full_sharpe"]:.3f}.

The one-week recommendation is a risk-controlled Momentum/MR corn sleeve with a Cargill physical disagreement filter.
"""
final_report = {"final_periods": final_periods, "comparison": final_comparison, "conclusion": conclusion}

display(Markdown("### Final candidate comparison"))
display(with_dd_pct(final_report["comparison"]))

display(Markdown("### Final selected row"))
display(with_dd_pct(pd.DataFrame([cargill["final_row"]]), [
    "base_strategy", "variant", "guard", "oos_sharpe", "oos_pnl", "oos_dd",
    "full_sharpe", "full_pnl", "full_dd", "turnover", "guard_oos_pct",
]))

display(Markdown("### Named-period check"))
display(with_dd_pct(final_report["final_periods"]))

display(Markdown(final_report["conclusion"]))


### Final candidate comparison

,strategy,variant,guard,oos_sharpe,oos_pnl,oos_dd_pct,full_sharpe,full_dd_pct
0,final_wf_momentum_mr_cargill,cargill_disagree_flat,below_ma_or_negative_mom_flat,1.413,215.627,-1.386,1.413,-1.386
1,wf_momentum_mr_base_reference,base_no_cargill,no_guard,0.488,643.137,-5.842,0.488,-5.842
2,guarded_mom20_benchmark,base_no_cargill,below_ma_or_negative_mom_flat,1.247,644.146,-3.698,0.542,-13.311


### Final selected row

,base_strategy,variant,guard,oos_sharpe,oos_pnl,oos_dd_pct,full_sharpe,full_pnl,full_dd_pct,turnover,guard_oos_pct
14,wf_momentum_mr,cargill_disagree_flat,below_ma_or_negative_mom_flat,1.413,215.627,-1.386,1.413,215.627,-1.386,0.028,0.780


### Named-period check

,period,total_pnl,sharpe,max_dd_pct,hit_rate,active_days,note
0,2010-2011: Russian drought/export ban shock,NaN,NaN,NaN,NaN,0,before first active trade (2018)
1,2012-2013: US drought rally/retrace,NaN,NaN,NaN,NaN,0,before first active trade (2018)
2,2014: Crimea/Black Sea shock,NaN,NaN,NaN,NaN,0,before first active trade (2018)
3,2014-2017: Low-price abundant supply,NaN,NaN,NaN,NaN,0,before first active trade (2018)
4,2018-2020: US-China trade war,200.076,3.216,-0.580,0.500,50,
5,2019: 2019 prevented planting floods,193.979,3.422,-0.394,0.500,44,
6,2020: COVID demand shock,NaN,NaN,NaN,NaN,0,strategy flat in this period
7,2020: COVID recovery/China buying,51.603,1.075,-1.386,0.522,23,



### Conclusion

**Core idea:** annual walk-forward Momentum/MR is the main tradable structure.

**Cargill use:** Cargill inventory/crush activity is used as a physical disagreement filter, not as the dominant corn alpha.

**Final corn candidate:** wf_momentum_mr with cargill_disagree_flat and guard below_ma_or_negative_mom_flat. OOS Sharpe 1.413, OOS PnL 215.627, OOS DD -1.39%, full-period Sharpe 1.413.

The one-week recommendation is a risk-controlled Momentum/MR corn sleeve with a Cargill physical disagreement filter.
